# CFEU journal extension: Notebook 6, intersectional fairness and latency

Run after Notebook 1. It takes about 5 minutes.

**What it measures**
1. **Intersectional fairness:** approval outcomes by age band × gender, for all applicants and thin-file applicants, under the bureau-only and combined models. It uses the same 80/20 split as Notebooks 3 to 5.
2. **Latency:** time to score one applicant and compute exact TreeSHAP reasons for them, and throughput for a batch of 10,000.

**Send back to Claude:** `nb6_results.json`, which downloads at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, platform
import numpy as np, pandas as pd
!pip -q install lightgbm pyarrow
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
OUT = '/content/drive/MyDrive/cfeu_journal'
p = f'{OUT}/home_credit_features.parquet'
hc = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv(f'{OUT}/home_credit_features.csv.gz')
y = hc['TARGET'].values
BUR = [c for c in hc.columns if c.startswith('bur_')]; ALT = [c for c in hc.columns if c.startswith('alt_')]; COMB = BUR + ALT
P = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=1.0, random_state=11, verbose=-1, n_jobs=-1)
idx_tr, idx_te = train_test_split(np.arange(len(hc)), test_size=0.2, stratify=y, random_state=0)
tr, te = hc.iloc[idx_tr], hc.iloc[idx_te]; ytr, yte = y[idx_tr], y[idx_te]
t0 = time.time(); m_comb = lgb.LGBMClassifier(**P).fit(tr[COMB], ytr); train_s = time.time() - t0
m_bur = lgb.LGBMClassifier(**P).fit(tr[BUR], ytr)
print(f'Trained in {train_s:.0f}s; test AUC combined {roc_auc_score(yte, m_comb.predict_proba(te[COMB])[:,1]):.3f}')

In [ ]:
# 1. Intersectional fairness: age band x gender
def cells_table(yv, p, groups, approve_rate=0.7):
    cut = np.quantile(p, approve_rate); ap = p <= cut; rows = {}
    for g in sorted(pd.unique(groups)):
        m = groups == g
        if m.sum() < 100: continue
        rows[g] = {'n': int(m.sum()), 'default_rate': float(yv[m].mean()), 'approval_rate': float(ap[m].mean()),
                   'approval_rate_good': float(ap[m & (yv == 0)].mean()),
                   'default_rate_approved': float(yv[m & ap].mean()) if (m & ap).any() else None}
    ref = max(rows, key=lambda k: rows[k]['approval_rate'])
    for g in rows:
        rows[g]['approval_ratio_vs_best'] = rows[g]['approval_rate'] / rows[ref]['approval_rate']
        rows[g]['good_gap_vs_best'] = rows[g]['approval_rate_good'] - rows[ref]['approval_rate_good']
    return {'reference_cell': ref, 'cells': rows}

grp = (te['audit_age_band'].astype(str) + ' | ' + te['audit_gender'].fillna('NA').astype(str)).values
thin = te['thin_file'].values == 1
p_c = m_comb.predict_proba(te[COMB])[:, 1]; p_b = m_bur.predict_proba(te[BUR])[:, 1]
inter = {}
for scope, mask in [('all', np.ones(len(te), bool)), ('thin_file', thin)]:
    inter[scope] = {'bureau': cells_table(yte[mask], p_b[mask], grp[mask]),
                    'combined': cells_table(yte[mask], p_c[mask], grp[mask])}
    print(scope, 'reference cell (combined):', inter[scope]['combined']['reference_cell'])
    print(pd.DataFrame({k: {'n': v['n'], 'approval': round(v['approval_rate'], 3), 'ratio': round(v['approval_ratio_vs_best'], 2),
                            'good_gap': round(v['good_gap_vs_best'], 3)} for k, v in inter[scope]['combined']['cells'].items()}).T)

In [ ]:
# 2. Latency: one applicant at a time, and a batch of 10,000
booster = m_comb.booster_
X = te[COMB]
one = [X.iloc[[i]] for i in range(1000)]
for r in one[:20]: booster.predict(r); booster.predict(r, pred_contrib=True)   # warm-up
def timed(fn, items):
    t = []
    for r in items:
        s = time.perf_counter(); fn(r); t.append((time.perf_counter() - s) * 1000)
    return {'median_ms': float(np.median(t)), 'p95_ms': float(np.percentile(t, 95)), 'p99_ms': float(np.percentile(t, 99))}
lat = {'score_only': timed(lambda r: booster.predict(r), one),
       'treeshap_only': timed(lambda r: booster.predict(r, pred_contrib=True), one),
       'score_plus_treeshap_plus_reasons': timed(lambda r: (booster.predict(r), np.argsort(-booster.predict(r, pred_contrib=True)[0, :-1])[:4]), one)}
B = X.iloc[:10000]
s = time.perf_counter(); booster.predict(B); bs = time.perf_counter() - s
s = time.perf_counter(); booster.predict(B, pred_contrib=True); bt = time.perf_counter() - s
lat['batch_10000'] = {'score_seconds': bs, 'treeshap_seconds': bt, 'applicants_per_second_with_treeshap': 10000 / (bs + bt)}
lat['environment'] = {'cpu_count': os.cpu_count(), 'processor': platform.processor() or platform.machine(),
                      'lightgbm': lgb.__version__, 'trees': booster.num_trees(), 'features': len(COMB), 'train_seconds': train_s}
print(json.dumps(lat, indent=1))

In [ ]:
res = {'intersectional': inter, 'latency': lat}
with open(f'{OUT}/nb6_results.json', 'w') as f: json.dump(res, f, indent=1)
from google.colab import files
files.download(f'{OUT}/nb6_results.json')
print('Send nb6_results.json to Claude.')